# Live Trading Health Diagnostic

**Use when:** no new signals in the Telegram `/signals` feed for 1+ hours, or you want to confirm the trader is healthy after a restart/deploy.

## What this notebook checks (read-only)

| Check | What it tells you |
|---|---|
| **Service status** | Is `ict-trader-live.service` active, stopped, or failed? |
| **Open packages** | Are there open order_packages blocking the BUG-046 gate? |
| **Recent signals** | Last 10 signals per strategy — confirm ticks are still firing |
| **Recent logs** | Last 60 journalctl lines from the trader service |
| **Health verdict** | Plain-English summary of what needs fixing |

## Fix paths (after running this notebook)

| Finding | Fix |
|---|---|
| Service stopped/failed | Run Cell 6 (restart service) |
| Open packages with **no** linked_trade_id | Run `sweep_unlinked_packages.ipynb` |
| Open packages **with** linked_trade_id | Trade is live on Bybit; wait for SL/TP, or use `/closeall <strategy>` |
| No signal gap — ticks firing normally | Market not generating setups; no action needed |

## Required Colab Secrets

| Name | What it holds |
|---|---|
| `VM_SSH_HOST` | VM hostname or public IP |
| `VM_SSH_USER` | SSH user (usually `ubuntu`) |

SSH key must be at `ICT_Bot_Secrets/ict-bot-ovm-private.key` on Google Drive (same as all other operator notebooks).

In [ ]:
# Cell 1: SSH setup (mirrors rotate_api_keys.ipynb pattern)
import os, shutil, stat, subprocess, tempfile
from google.colab import drive, userdata

print('⏳ Mounting Google Drive…')
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'⚠️  drive.mount() raised: {exc}')

DEFAULT_SSH_KEY_NAMES = [
    'ict-bot-ovm-private.key',
    'vm_ssh_key', 'id_rsa', 'id_ed25519', 'id_ecdsa',
]
DRIVE_FOLDER = '/content/drive/MyDrive/ICT_Bot_Secrets'

ssh_key_path = None
if os.path.exists('/content/drive/MyDrive'):
    for name in DEFAULT_SSH_KEY_NAMES:
        path = os.path.join(DRIVE_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            break
if ssh_key_path is None:
    raise SystemExit(
        f'No SSH key found in {DRIVE_FOLDER}/. Place '
        '`ict-bot-ovm-private.key` there and re-run.'
    )
print(f'✅ SSH key located: {ssh_key_path}')

host = userdata.get('VM_SSH_HOST')
user = userdata.get('VM_SSH_USER')
if not host or not user:
    raise SystemExit('VM_SSH_HOST + VM_SSH_USER must be set in Colab Secrets.')

tmpdir = tempfile.mkdtemp(prefix='diag-live-')
key_path = os.path.join(tmpdir, 'vm_key')
shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)

ssh_target = f'{user}@{host}'
ssh_opts = [
    '-i', key_path,
    '-o', 'StrictHostKeyChecking=accept-new',
    '-o', 'UserKnownHostsFile=' + os.path.join(tmpdir, 'known_hosts'),
    '-o', 'ConnectTimeout=15',
    '-o', 'BatchMode=yes',
]

def run_ssh(cmd, *, label, allow_fail=False):
    proc = subprocess.run(
        ['ssh'] + ssh_opts + [ssh_target, cmd],
        capture_output=True, text=True,
    )
    if proc.returncode != 0 and not allow_fail:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print((proc.stderr or proc.stdout)[:500])
        raise SystemExit(f'{label} failed')
    return (proc.stdout or '').strip()

print(f'⏳ Connecting to {ssh_target} …')
run_ssh('echo connected', label='SSH connectivity')
print('✅ SSH OK')

DB_PATH = f'/home/{user}/ict-trading-bot/trade_journal.db'
SERVICE = 'ict-trader-live.service'

In [ ]:
# Cell 2: Service status
print('=' * 60)
print('CHECK 1 — Service status')
print('=' * 60)

active = run_ssh(f'systemctl is-active {SERVICE}', label='is-active', allow_fail=True)
enabled = run_ssh(f'systemctl is-enabled {SERVICE}', label='is-enabled', allow_fail=True)

status_icon = '✅' if active == 'active' else '🔴'
print(f'{status_icon} {SERVICE}: {active} (enabled={enabled})')

if active != 'active':
    print()
    print('⚠️  Service is NOT running. Run Cell 6 to restart it.')
    # Show last status for context
    status_out = run_ssh(
        f'systemctl status {SERVICE} --no-pager -l 2>&1 | head -30',
        label='status detail', allow_fail=True,
    )
    print(status_out)
else:
    uptime = run_ssh(
        f'systemctl show {SERVICE} --property=ActiveEnterTimestamp --value',
        label='uptime', allow_fail=True,
    )
    print(f'   Running since: {uptime}')

In [ ]:
# Cell 3: Open order packages per strategy (the BUG-046 gate query)
print()
print('=' * 60)
print('CHECK 2 — Open order packages (BUG-046 gate state)')
print('=' * 60)

summary_sql = (
    "SELECT strategy_name, "
    "COUNT(*) AS total_open, "
    "SUM(CASE WHEN linked_trade_id IS NULL THEN 1 ELSE 0 END) AS unlinked, "
    "SUM(CASE WHEN linked_trade_id IS NOT NULL THEN 1 ELSE 0 END) AS linked, "
    "MIN(created_at) AS oldest "
    "FROM order_packages "
    "WHERE status = 'open' "
    "GROUP BY strategy_name "
    "ORDER BY total_open DESC;"
)
summary = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{summary_sql}"',
    label='open packages summary',
)
print(summary or '✅ No open packages — gate is clear for all strategies.')

if summary:
    print()
    print('Full open package detail:')
    detail_sql = (
        "SELECT order_package_id, strategy_name, symbol, direction, "
        "status, linked_trade_id, created_at, updated_at "
        "FROM order_packages WHERE status = 'open' "
        "ORDER BY strategy_name, datetime(created_at) DESC;"
    )
    detail = run_ssh(
        f'sqlite3 -header -column {DB_PATH} "{detail_sql}"',
        label='detail rows',
    )
    print(detail)
    print()
    print('Fix guidance:')
    print('  • unlinked > 0  → run sweep_unlinked_packages.ipynb (no trade was ever placed)')
    print('  • linked > 0    → trade is live on Bybit; wait for SL/TP or use /closeall <strategy>')

In [ ]:
# Cell 4: Recent signals per strategy
print()
print('=' * 60)
print('CHECK 3 — Recent signals (last 10 per strategy)')
print('=' * 60)

signals_sql = (
    "SELECT strategy_name, timestamp, symbol, direction, signal_type, reason "
    "FROM signals "
    "ORDER BY datetime(timestamp) DESC "
    "LIMIT 20;"
)
signals = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{signals_sql}" 2>/dev/null',
    label='recent signals', allow_fail=True,
)
print(signals or '(signals table empty or not found)')

print()
last_signal_ts = run_ssh(
    f'sqlite3 {DB_PATH} "SELECT MAX(timestamp) FROM signals;" 2>/dev/null',
    label='last signal ts', allow_fail=True,
)
print(f'Last signal timestamp: {last_signal_ts or "(unknown)"}')

In [ ]:
# Cell 5: Recent journalctl logs
print()
print('=' * 60)
print('CHECK 4 — Recent trader service logs (last 60 lines)')
print('=' * 60)

logs = run_ssh(
    f'journalctl -u {SERVICE} -n 60 --no-pager 2>&1',
    label='journalctl', allow_fail=True,
)
print(logs or '(no logs found)')

print()
print('=' * 60)
print('MONITOR RECONCILER — check reconciler is enabled')
print('=' * 60)
reconcile_lines = run_ssh(
    f'journalctl -u {SERVICE} -n 500 --no-pager 2>&1 | grep -i reconcil | tail -10',
    label='reconciler logs', allow_fail=True,
)
print(reconcile_lines or '(no reconciler log lines found — check MONITOR_RECONCILE_ENABLED in .env)')

In [ ]:
# Cell 6: RESTART SERVICE (only run if Cell 2 showed service is stopped/failed)
# Set RESTART = True after confirming the service is not running.
RESTART = False  # <- set to True to restart

if not RESTART:
    print('RESTART is False — no action taken.')
    print('Set RESTART = True and re-run this cell if the service is not active.')
else:
    print(f'⏳ Restarting {SERVICE} …')
    run_ssh(f'sudo systemctl restart {SERVICE}', label='restart')
    import time; time.sleep(5)
    new_status = run_ssh(f'systemctl is-active {SERVICE}', label='post-restart status', allow_fail=True)
    icon = '✅' if new_status == 'active' else '🔴'
    print(f'{icon} Post-restart status: {new_status}')
    if new_status == 'active':
        print()
        print('Service is running. New signals should appear within one tick interval.')
        print('Watch Telegram /signals or journalctl -f -u ict-trader-live.service')
    else:
        print()
        print('Service failed to start. Check logs:')
        fail_logs = run_ssh(
            f'journalctl -u {SERVICE} -n 30 --no-pager 2>&1',
            label='fail logs', allow_fail=True,
        )
        print(fail_logs)

In [ ]:
# Cell 7: Health verdict summary
import shutil as _shutil
_shutil.rmtree(tmpdir, ignore_errors=True)

print()
print('=' * 60)
print('HEALTH VERDICT')
print('=' * 60)
print()
print('Review Cell 2 (service), Cell 3 (open packages), Cell 5 (logs).')
print()
print('Most common signal-gap causes and fixes:')
print()
print('1. Service stopped/failed')
print('   → Set RESTART=True in Cell 6 and re-run')
print()
print('2. Unlinked open packages (no linked_trade_id)')
print('   → Run notebooks/operator/sweep_unlinked_packages.ipynb')
print('   → Gate clears on next tick; no restart needed')
print()
print('3. Linked open packages (trade active on Bybit)')
print('   → Wait for SL/TP to hit, or send /closeall <strategy> in Telegram')
print('   → The monitor loop reconciles every tick; Bybit holds SL/TP at broker')
print()
print('4. No gap — last signal timestamp is recent')
print('   → Market not generating setups for this strategy; no action needed')
print()
print('5. MONITOR_RECONCILE_ENABLED missing from .env')
print('   → Check .env on VM: grep MONITOR_RECONCILE_ENABLED ~/ict-trading-bot/.env')
print('   → Must be true. If missing, add it and restart the service.')